# Streaming Response With In-Text Citations

**Author:** Shinin Varongchayakul

**Date:** 23 Jul 2026

## 1. Get Retrieved Documents

In [31]:
# Set retrieved documents
retrieved_docs = [
    {
        "id": "hr_0001",
        "content": "Employees are entitled to 12 days of annual leave after completing one year of service.",
        "metadata": {
            "source": "Employee Handbook.pdf",
            "author": "HR Department",
            "date": "2025-01-15"
        }
    },
    {
        "id": "hr_0002",
        "content": "Employees may work remotely up to two days per week with manager approval.",
        "metadata": {
            "source": "Remote Work Policy.pdf",
            "author": "HR Department",
            "date": "2025-03-10"
        }
    },
    {
        "id": "hr_0003",
        "content": "Travel expenses must be submitted within 30 days after the business trip.",
        "metadata": {
            "source": "Expense Policy.pdf",
            "author": "Finance Department",
            "date": "2024-11-01"
        }
    }
]

## 2. Set Response Generator

### 2.1 Prompt

In [32]:
# Set system prompt
system_prompt = """
You are an AI assistant that answers user questions using ONLY the documents provided below.

Cite every claim inline using the document's ID exactly as given, wrapped in square brackets (e.g., [hr_0010]).

Only cite IDs you actually use to answer the question. Do not invent IDs.
"""

# Set user prompt
user_prompt = """
Documents:
{docs}

User's query:
{query}
"""

In [33]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Set prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt.strip()),
        ("user", user_prompt.strip())
    ]
)

### 2.2 LLM

In [34]:
# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY_01")

In [35]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    api_key=GEMINI_API_KEY
)

### 2.3 Chain Pipeline

In [36]:
# Import package
from langchain_core.output_parsers import StrOutputParser

# Create chain pipeline
chain = prompt_template | llm | StrOutputParser()

## 3. Create Functions

### 3.1 Document Formatter

In [37]:
# Format documents
def format_docs(docs: list[dict]) -> str:

    # Instantiate collector
    formatted = []

    # Loop through documents
    for doc in docs:
        metadata = doc.get("metadata", {})
        formatted.append(
            f"ID: {doc['id']}\n"
            f"Content: {doc['content']}\n"
            f"Source: {metadata.get('source', 'N/A')}\n"
            "-------------------------"
        )

    # Return formatted string
    return "\n".join(formatted)

### 3.2 Streaming Response Generator

In [ ]:
# Define function to stream response with citations
def stream_response_with_citations(query: str, docs: list[dict]) -> None:

    # Set regex pattern for citations
    CITE_RE = re.compile(r"\[([A-Za-z0-9_\-]+)\]")

    # Build lookup dicts
    docs_by_id = {doc["id"]: doc for doc in docs}

    # Set required variables
    citation_numbers: dict[str, int] = {}
    ordered_citations: list[dict] = []
    buffer = ""

    # Loop through streaming
    for chunk in chain.stream(
        {
            "docs": format_docs(docs),
            "query": query
        }
    ):

        # Add chunk to buffer
        buffer += chunk
        output = ""

        # Replace match ID with display number
        while True:
            match = CITE_RE.search(buffer)
            if not match:
                break

            # Text before the match is safe to flush as-is
            output += buffer[:match.start()]

            doc_id = match.group(1)

            # Assign display number to match citations
            if doc_id in docs_by_id:
                if doc_id not in citation_numbers:
                    citation_numbers[doc_id] = len(citation_numbers) + 1
                    ordered_citations.append(docs_by_id[doc_id])
                output += f"[{citation_numbers[doc_id]}]"
            else:
                output += match.group(0)

            # Drop consumed portion from the buffer
            buffer = buffer[match.end():]

        # Re-scan for citations splitted across chunks
        last_bracket = buffer.rfind("[")
        if last_bracket != -1:
            output += buffer[:last_bracket]
            buffer = buffer[last_bracket:]
        else:
            output += buffer
            buffer = ""

        if output:
            yield {
                "type": "token",
                "content": output
            }

    # Flush any leftover
    if buffer:
        yield {
            "type": "token",
            "content": buffer
        }

    # Yield citations
    yield {
        "type": "citation",
        "content": ordered_citations
    }

### 3.3 Response Printer

In [39]:
# Define function to print streaming response
def print_streamed_response(event_stream) -> list[dict]:

    # Set default in case no citation event is ever yielded
    ordered_citations: list[dict] = []

    # Loop through yielded events
    for event in event_stream:

        # Check if event is response text
        if event["type"] == "token":
            print(
                event["content"],
                end="",
                flush=True
            )

        # Check if event is citations
        elif event["type"] == "citation":
            ordered_citations = event["content"]
            print("\n\n--- Sources ---")

            for number, c in enumerate(ordered_citations, start=1):
                print(f"[{number}] {c['metadata']['source']}")

    # Return citations
    return ordered_citations

## 4. Generate Response

In [40]:
# Set user query
user_query = "How many annual leave days do employees receive, and can they work remotely?"

# Generate response
citations = print_streamed_response(
    stream_response_with_citations(
        query=user_query,
        docs=retrieved_docs
    )
)

Employees are entitled to 12 days of annual leave after completing one year of service [1]. They may work remotely up to two days per week with manager approval [2].

--- Sources ---
[1] Employee Handbook.pdf
[2] Remote Work Policy.pdf
